In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

np.random.seed(42)  

print("Libraries loaded!")
print("pandas:", pd.__version__)
print("numpy:", np.__version__)

Libraries loaded!
pandas: 3.0.1
numpy: 2.4.2


In [2]:
# How many restaurants to simulate --- Merchant (Restaurant) Data
n_merchants = 500

merchants = pd.DataFrame({
    
    'merchant_id': range(n_merchants),   # 0, 1, 2, 3 ... 499
    
    'city': np.random.choice(
        ['Mumbai', 'Delhi', 'Bangalore', 'Hyderabad', 'Chennai', 'Pune'], 
        n_merchants
    ),
    
    'cuisine': np.random.choice(
        ['North Indian', 'South Indian', 'Chinese', 'Fast Food', 'Biryani'], 
        n_merchants
    ),
    
    'size': np.random.choice(
        ['Small', 'Medium', 'Large'], 
        n_merchants, 
        p=[0.5, 0.35, 0.15]   # 50% small, 35% medium, 15% large
    ),
    
    # Average prep time for this restaurant (in minutes)
    'avg_prep_time': np.random.uniform(8, 40, n_merchants),
    
    # KEY COLUMN: How reliably does this restaurant press FOR on time?
    # 0 = very unreliable, 1 = perfectly reliable
    # beta(2,5) gives mostly low values — most restaurants are unreliable
    'for_reliability': np.random.beta(2, 5, n_merchants),
    
    # How many Zomato orders per day on average
    'daily_zomato_orders': np.random.poisson(20, n_merchants),
    
    # Total orders including Swiggy + dine-in (Zomato can't see this!)
    'daily_total_orders': np.random.poisson(50, n_merchants),
    
    # Number of cooking stations in the kitchen
    'kitchen_stations': np.random.choice([1, 2, 3, 4, 5], n_merchants, 
                                          p=[0.2, 0.3, 0.25, 0.15, 0.1])
})

# Always check your data after creating it!
print("Shape:", merchants.shape)   # (rows, columns)
print("\nFirst 5 rows:")
merchants.head()

Shape: (500, 9)

First 5 rows:


,merchant_id,city,cuisine,size,avg_prep_time,for_reliability,daily_zomato_orders,daily_total_orders,kitchen_stations
0,0,Hyderabad,North Indian,Medium,38.195709,0.377940,24,45,1
1,1,Chennai,Biryani,Medium,21.431274,0.050302,16,54,3
2,2,Bangalore,South Indian,Medium,28.432830,0.133127,28,63,4
3,3,Chennai,South Indian,Small,20.723021,0.298860,19,57,5
4,4,Chennai,Biryani,Medium,16.774886,0.320863,26,36,4


In [3]:
# Get a statistical summary of every column --- Merchant Data
print("=== MERCHANT DATA SUMMARY ===")
print(merchants.describe().round(2))

print("\n=== HOW MANY RESTAURANTS PER CITY? ===")
print(merchants['city'].value_counts())

print("\n=== FOR RELIABILITY DISTRIBUTION ===")
low  = (merchants['for_reliability'] < 0.3).sum()
mid  = ((merchants['for_reliability'] >= 0.3) & (merchants['for_reliability'] < 0.7)).sum()
high = (merchants['for_reliability'] >= 0.7).sum()
print(f"Unreliable restaurants (score < 0.3): {low}  ({low/n_merchants*100:.0f}%)")
print(f"Average restaurants   (0.3 to 0.7):  {mid}  ({mid/n_merchants*100:.0f}%)")
print(f"Reliable restaurants  (score > 0.7): {high}  ({high/n_merchants*100:.0f}%)")

=== MERCHANT DATA SUMMARY ===
       merchant_id  avg_prep_time  for_reliability  daily_zomato_orders  \
count       500.00         500.00           500.00               500.00   
mean        249.50          23.94             0.28                19.95   
std         144.48           9.31             0.15                 4.68   
min           0.00           8.16             0.01                 9.00   
25%         124.75          15.54             0.16                17.00   
50%         249.50          24.02             0.26                20.00   
75%         374.25          31.42             0.37                23.00   
max         499.00          39.98             0.77                33.00   

       daily_total_orders  kitchen_stations  
count              500.00            500.00  
mean                49.86              2.67  
std                  7.14              1.27  
min                 33.00              1.00  
25%                 45.00              2.00  
50%               

In [4]:
# Order Data

n_orders = 50000

orders = pd.DataFrame({
    
    'order_id': range(n_orders),
    
    # Each order belongs to a random restaurant
    'merchant_id': np.random.choice(merchants['merchant_id'], n_orders),
    
    # Number of items in the order
    'num_items': np.random.choice([1, 2, 3, 4, 5], n_orders, 
                                   p=[0.3, 0.35, 0.2, 0.1, 0.05]),
    
    'order_value': np.random.uniform(100, 1500, n_orders),
    
    # Is this order during a busy period? (lunch 12-2pm, dinner 7-10pm)
    'is_peak_hour': np.random.choice([0, 1], n_orders, p=[0.6, 0.4]),
    
    'is_weekend': np.random.choice([0, 1], n_orders, p=[0.7, 0.3]),
})

# Bring in restaurant-level columns (join like Excel VLOOKUP)
orders = orders.merge(
    merchants[['merchant_id', 'avg_prep_time', 'for_reliability', 'daily_total_orders']], 
    on='merchant_id'
)

print("Orders shape:", orders.shape)
print("\nFirst 3 rows:")
orders.head(3)

Orders shape: (50000, 9)

First 3 rows:


,order_id,merchant_id,num_items,order_value,is_peak_hour,is_weekend,avg_prep_time,for_reliability,daily_total_orders
0,0,296,2,808.682714,0,0,37.665105,0.076945,56
1,1,111,2,1211.906824,0,0,39.193381,0.529800,50
2,2,449,3,1090.809168,0,0,35.777814,0.229863,43


In [6]:
# ================================================================
# ACTUAL KPT — The REAL prep time (what we WISH we had as labels)
# ================================================================
# Formula: base prep time × peak hour penalty × more items = longer
orders['actual_kpt'] = (
    orders['avg_prep_time']                          # restaurant's base speed
    * (1 + 0.3  * orders['is_peak_hour'])            # 30% slower during peak
    * (1 + 0.08 * (orders['num_items'] - 1))         # 8% longer per extra item
    * (1 + 0.02 * orders['daily_total_orders'] / 10) # busier kitchen = slower
    + np.random.normal(0, 3, n_orders)               # natural randomness
).clip(3, 60)  # Keep between 3 and 60 minutes (realistic range)


# ================================================================
# FOR-BASED KPT — What Zomato's system actually records (CORRUPTED)
# ================================================================
# Unreliable restaurants add a lot of noise (they press late)
# Reliable restaurants add very little noise

noise = np.where(
    orders['for_reliability'] < 0.3,
    # Unreliable: FOR is pressed ~5 min late on average, with high variance
    np.random.normal(5, 4, n_orders),
    # Reliable: FOR is mostly accurate, tiny error
    np.random.normal(0.5, 1, n_orders)
)

orders['for_kpt'] = (orders['actual_kpt'] + noise).clip(1, 60)

# The ERROR = how wrong the label is
orders['label_error'] = orders['for_kpt'] - orders['actual_kpt']


# ================================================================
# RIDER DATA
# ================================================================
orders['rider_travel_time'] = np.random.uniform(3, 15, n_orders)

# Rider wait = how long rider sits at restaurant waiting for food
# (actual prep time) - (time that passed before rider arrived)
orders['rider_wait'] = np.maximum(
    0,   # can't be negative
    orders['actual_kpt'] - (orders['rider_travel_time'] + np.random.uniform(-2, 2, n_orders))
)

print("=== KEY NUMBERS ===")
print(f"Avg ACTUAL KPT:      {orders['actual_kpt'].mean():.1f} min")
print(f"Avg FOR-based KPT:   {orders['for_kpt'].mean():.1f} min")
print(f"Avg label error:     {orders['label_error'].mean():.1f} min")
print(f"Avg rider wait:      {orders['rider_wait'].mean():.1f} min")
print(f"\nOrders where label is off by >5 min: {(abs(orders['label_error']) > 5).sum()} ({(abs(orders['label_error']) > 5).mean()*100:.0f}%)")

=== KEY NUMBERS ===
Avg ACTUAL KPT:      32.4 min
Avg FOR-based KPT:   35.3 min
Avg label error:     2.9 min
Avg rider wait:      23.5 min

Orders where label is off by >5 min: 13648 (27%)


In [7]:
# Save to CSV files in your project folder
merchants.to_csv('merchants.csv', index=False)
orders.to_csv('orders.csv', index=False)

print("✅ Files saved!")
print(f"   merchants.csv  →  {len(merchants)} rows, {len(merchants.columns)} columns")
print(f"   orders.csv     →  {len(orders)} rows, {len(orders.columns)} columns")

# Quick sanity check — reload and verify
test = pd.read_csv('orders.csv')
print(f"\nReload test: {len(test)} rows loaded back correctly ✅")

✅ Files saved!
   merchants.csv  →  500 rows, 9 columns
   orders.csv     →  50000 rows, 14 columns

Reload test: 50000 rows loaded back correctly ✅
